In [1]:
import os
import json
import torch

from PIL import Image, ImageDraw
from pdf2image import convert_from_path
import pandas as pd

from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
from surya.layout import LayoutPredictor


from transformers import AutoTokenizer, AutoModelForCausalLM
from surya.recognition import RecognitionPredictor


from tqdm import tqdm
import time
import lmstudio as lms

/home/duckq1u/miniconda3/envs/OCR2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def draw_bounding_boxes(image, predictions, is_layout=False):
    draw = ImageDraw.Draw(image)
    # Rounded rectangle parameters
    border_radius = 0  # Corner radius
    outline_color = "red"  # Box color
    outline_width = 1  # Box thickness
    print(predictions)

    if is_layout:
        predictions = predictions[0].bboxes
    else:
        predictions = predictions[0].text_lines

    paragraph = ""

    # Loop through each set of coordinates
    for coords in predictions:
        # Extract x and y values
        x_values = [x for x, y in coords.polygon]
        y_values = [y for x, y in coords.polygon]

        # Calculate bounding box
        min_x = min(x_values)
        max_x = max(x_values)
        min_y = min(y_values)
        max_y = max(y_values)

        # Draw the rounded rectangle
        draw.rounded_rectangle(
            [(min_x, min_y), (max_x, max_y)],
            radius=border_radius,
            outline=outline_color,
            width=outline_width,
        )
        if is_layout is False:
            paragraph += coords.text + "\n"
    return image, paragraph


def rounding_box(image, is_layout=False):
    image_cp = image.copy()
    if is_layout:
        layout_predictor = LayoutPredictor()
        detection_predictor = DetectionPredictor()
        predictions = layout_predictor([image_cp])
    else:
        langs = [
            "en"
        ]  # Replace with your languages or pass None (recommended to use None)
        recognition_predictor = RecognitionPredictor()
        detection_predictor = DetectionPredictor()
        predictions = recognition_predictor(
            [image_cp], det_predictor=detection_predictor
        )

    image_output, paragraph = draw_bounding_boxes(image_cp, predictions, is_layout)
    return image_output, paragraph


SERVER_API_HOST = "localhost:8501"

# This must be the *first* convenience API interaction (otherwise the SDK
# implicitly creates a client that accesses the default server API host)
lms.configure_default_client(SERVER_API_HOST)


model = lms.llm("gemma-3-12b-it-qat", config={"seed": 5315813802286225740, "flashAttention": True, "evalBatchSize": 7024, "contextLength": 7024})

In [3]:
import re
def norm_text(text):
    # remove :, punctuation, and special characters
    text = re.sub(r"[^\w\s]", " ", text)  # replace punctuation and special characters with space
    text = re.sub(r"\s+", " ", text)  # replace multiple spaces with a single space
    return text.strip()

In [ ]:


folder_path = '/home/duckq1u/Documents/obsidian_aio/Notebook/Dự án/OCR anh hiếu/OCR/Attachments/PDF_form_list'
json_path = os.path.join(folder_path, "infor.json")

total_time = 0
count = 0

for filename in sorted(os.listdir(folder_path)):
    count += 1
    start_time = time.time()
    
    if not filename.endswith('.pdf'):
        continue
    print(f"Đang xử lý file: {filename}")
    file_path = os.path.join(folder_path, filename)

    images = convert_from_path(file_path)
    # image = images[0]
    # image.save('/home/duckq1u/Documents/obsidian_aio/Notebook/Môn học trên trường/OCR thầy minh/OCR/Attachments/PDF_form_list/8004110070.jpg', 'JPEG')
    # NOTE: Prepare images for the model
    images_list = []
    paragraphs = []
    keywords = [
        'Người yêu cầu cấp bản sao/Applicant',
        'Yêu  cầu  cấp  bản  sao  Văn  bản  chứng  nhận  đăng  ký/Request  for  issuance  of  copy  of registration certificate']
    for index, image in enumerate(images):
        count = 0
        _, raw_text = rounding_box(image, is_layout=False)
        for key in keywords:
            if norm_text(key).lower() in norm_text(raw_text).lower():
                count += 1
        if count < 2: continue
        
        paragraph = f"\n------------ Trang {index + 1} ------\n"
        paragraph += raw_text + f"\n ------------ Kết thúc trang {index + 1} ------\n"
        image.save('./temp.jpg', 'JPEG')
        paragraphs.append(paragraph)
        image_handle = lms.prepare_image(
            "./temp.jpg",
        )
        images_list.append(image_handle)
    
    # Continue if no keyword are found in any image on file
    if len(images_list) == 0: continue
    
    raw_text = ""
    for paragraph in paragraphs:
        raw_text += paragraph + "\n"

    prompt = f"""
    You are an expert in administrative document processing

    <Raw text from image> 
    {raw_text}
    </Raw text from image>
    

    <Notes>
    1. Only output the required extracted information.
    2. Related information is often located near the extracted information and must also be captured.
    3. Perform detailed analysis and do not omit any information.
    4. Pay attention to the types of information checked box and extract the checked information.
    </Notes>
    
    <Requirements>
    Keyword list:
    1. 'Người yêu cầu cấp bản sao/Applicant'
    2. 'Yêu  cầu  cấp  bản  sao  Văn  bản  chứng  nhận  đăng  ký/Request  for  issuance  of  copy  of registration certificate'

    Step 1: Extract information from the IMAGE, using KEYWORDS as markers, starting from left to right and top to bottom until encountering another KEYWORD or unrelated information.

    Step 1.1: If the image contains table information related to the any KEYWORD, analyze the table as a list for subsequent retrieval, then load it into the JSON.

    Step 2: Extract information from the RAW TEXT FROM IMAGE using KEYWORDS as markers, starting from left to right and top to bottom until encountering another KEYWORD or unrelated information.
    
    Step 3: Use information extracted from IMAGE to correct spelling errors in the RAW TEXT FROM IMAGE.

    Step 4: Re-analyze the information and correct the spelling of the information before extracting.

    Step 5: Load the data into the following JSON form:
    ```JSON
    {{
    "Người yêu cầu cấp bản sao/Applicant": [ <extracted and related information that can have one or multiple. Pay attention to the types of information checked box and extract the checked information. All <information> inside the JSON must have a KEY and must not stand alone> ],
    "Yêu  cầu  cấp  bản  sao  Văn  bản  chứng  nhận  đăng  ký/Request  for  issuance  of  copy  of registration certificate": [ <extracted and related information that can have one or multiple. Pay attention to the types of information checked box and extract the checked information. All <information> inside the JSON must have a KEY and must not stand alone> ],
    }}
    ```
    While filling in the JSON, ensure that:
    1. KEYs with the same or equivalent meanings can be grouped into a list.
    2. Do not alter the format or structure of the JSON.
    3. If the <information> is in the form of a table, it should be stored in a list with column headers as titles.
    4. Merge objects with the same value.
    5. Remove duplicate data fields before output.

    Step 5: Output in JSON file format.
    </Requirements>"""
    # Chuẩn bị ảnh


    # Gửi request chat
    chat = lms.Chat("Bạn là một chuyên gia về xử lý tài liệu hành chính.")
    chat.add_user_message(prompt, images=images_list)

    prediction = model.respond(chat)


    duration = time.time() - start_time
    total_time += duration
    
    print("==" * 40)
    print("== Kết quả dự đoán ==")
    print(prediction)
    
    print(f"Thời gian xử lý: {duration:.2f}s")
    print("==" * 40)
    # print()
    # break
    
# ----- Tổng kết -----
avg_time = total_time / count
print(f"\n Thời gian trung bình mỗi file: {avg_time:.2f} giây ({total_time:.2f}s tổng cộng)")

Đang xử lý file: 1587366867.pdf


Recognizing Text: 100%|██████████| 35/35 [00:05<00:00,  6.52it/s]


[OCRResult(text_lines=[TextLine(polygon=[[1346.0, 81.0], [1524.0, 81.0], [1524.0, 113.0], [1346.0, 113.0]], confidence=0.9814565479755402, text='Mẫu số 05d', chars=[TextChar(polygon=[[1353.0, 81.0], [1386.0, 81.0], [1386.0, 112.0], [1353.0, 112.0]], confidence=0.9741962552070618, text='M', bbox_valid=True, bbox=[1353.0, 81.0, 1386.0, 112.0]), TextChar(polygon=[[1384.0, 81.0], [1401.0, 81.0], [1401.0, 112.0], [1384.0, 112.0]], confidence=0.9989340901374817, text='ẫ', bbox_valid=True, bbox=[1384.0, 81.0, 1401.0, 112.0]), TextChar(polygon=[[1400.0, 81.0], [1418.0, 81.0], [1418.0, 112.0], [1400.0, 112.0]], confidence=0.9911706447601318, text='u', bbox_valid=True, bbox=[1400.0, 81.0, 1418.0, 112.0]), TextChar(polygon=[[1417.0, 82.0], [1425.0, 82.0], [1425.0, 112.0], [1417.0, 112.0]], confidence=0.975827693939209, text=' ', bbox_valid=True, bbox=[1417.0, 82.0, 1425.0, 112.0]), TextChar(polygon=[[1425.0, 81.0], [1437.0, 81.0], [1437.0, 112.0], [1425.0, 112.0]], confidence=0.9896509647369385, 

Recognizing Text: 100%|██████████| 48/48 [00:02<00:00, 21.63it/s]


[OCRResult(text_lines=[TextLine(polygon=[[114.0, 79.0], [720.0, 79.0], [720.0, 115.0], [114.0, 115.0]], confidence=0.9532229120914752, text='CHI TIẾT THỐNG TIN ĐĂNG KÝ', chars=[TextChar(polygon=[[120.0, 79.0], [147.0, 79.0], [147.0, 114.0], [120.0, 114.0]], confidence=0.6190757155418396, text='C', bbox_valid=True, bbox=[120.0, 79.0, 147.0, 114.0]), TextChar(polygon=[[145.0, 79.0], [175.0, 79.0], [175.0, 114.0], [145.0, 114.0]], confidence=0.9926995635032654, text='H', bbox_valid=True, bbox=[145.0, 79.0, 175.0, 114.0]), TextChar(polygon=[[176.0, 79.0], [190.0, 79.0], [190.0, 114.0], [176.0, 114.0]], confidence=0.986676812171936, text='I', bbox_valid=True, bbox=[176.0, 79.0, 190.0, 114.0]), TextChar(polygon=[[190.0, 79.0], [199.0, 79.0], [199.0, 114.0], [190.0, 114.0]], confidence=0.9898042678833008, text=' ', bbox_valid=True, bbox=[190.0, 79.0, 199.0, 114.0]), TextChar(polygon=[[199.0, 79.0], [225.0, 79.0], [225.0, 114.0], [199.0, 114.0]], confidence=0.9862843751907349, text='T', bbox_v

...

[OCRResult(text_lines=[TextLine(polygon=[[1346.0, 82.0], [1523.0, 82.0], [1523.0, 112.0], [1346.0, 112.0]], confidence=0.9732641577720642, text='Mẫu số 05d', chars=[TextChar(polygon=[[1350.0, 82.0], [1385.0, 82.0], [1385.0, 111.0], [1350.0, 111.0]], confidence=0.9674336910247803, text='M', bbox_valid=True, bbox=[1350.0, 82.0, 1385.0, 111.0]), TextChar(polygon=[[1382.0, 82.0], [1399.0, 82.0], [1399.0, 111.0], [1382.0, 111.0]], confidence=0.9985573887825012, text='ẫ', bbox_valid=True, bbox=[1382.0, 82.0, 1399.0, 111.0]), TextChar(polygon=[[1399.0, 82.0], [1417.0, 82.0], [1417.0, 111.0], [1399.0, 111.0]], confidence=0.9907772541046143, text='u', bbox_valid=True, bbox=[1399.0, 82.0, 1417.0, 111.0]), TextChar(polygon=[[1416.0, 82.0], [1425.0, 82.0], [1425.0, 111.0], [1416.0, 111.0]], confidence=0.9789724946022034, text=' ', bbox_valid=True, bbox=[1416.0, 82.0, 1425.0, 111.0]), TextChar(polygon=[[1425.0, 82.0], [1437.0, 82.0], [1437.0, 111.0], [1425.0, 111.0]], confidence=0.9897791743278503,

Recognizing Text:   0%|          | 0/54 [00:00<?, ?it/s]

KeyboardInterrupt: 

{"event": "Websocket failed, terminating session.", "ws_url": "ws://localhost:8501/llm"}
Traceback (most recent call last):
  File "/home/duckq1u/miniconda3/envs/OCR2/lib/python3.10/site-packages/lmstudio/_ws_impl.py", line 475, in _receive_messages
    await self._process_next_message()
  File "/home/duckq1u/miniconda3/envs/OCR2/lib/python3.10/site-packages/lmstudio/_ws_impl.py", line 466, in _process_next_message
    message = await ws.receive_json()
  File "/home/duckq1u/miniconda3/envs/OCR2/lib/python3.10/site-packages/httpx_ws/_api.py", line 951, in receive_json
    data = await self.receive_text(timeout)
  File "/home/duckq1u/miniconda3/envs/OCR2/lib/python3.10/site-packages/httpx_ws/_api.py", line 861, in receive_text
    event = await self.receive(timeout)
  File "/home/duckq1u/miniconda3/envs/OCR2/lib/python3.10/site-packages/httpx_ws/_api.py", line 821, in receive
    raise event
httpx_ws._exceptions.WebSocketNetworkError
Stack (most recent call last):
  File "/home/duckq1u/m

: 